## Импорты и конфиг

In [1]:
!pip install -q transformers accelerate sentencepiece
!pip install -q bert-score sacrebleu
!pip install -q "trl[peft]" bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 528.8/528.8 kB 22.2 MB/s eta 0:00:00


In [2]:
import torch
import torch.nn.functional as F
import numpy as np
import random
import os
import json
import zipfile
from tqdm import tqdm

# Transformers
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

# Datasets и TRL
from datasets import Dataset
from trl import GRPOTrainer, GRPOConfig

# PEFT для LoRA
from peft import LoraConfig, get_peft_model, PeftModel

# Метрики
from bert_score import BERTScorer
import sacrebleu

# Визуализация
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Google Colab (если нужно)
from google.colab import files as colab_files

In [3]:
def set_seed(seed=42):
    """
    Установка seed для воспроизводимости результатов
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    # Для детерминированных вычислений
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    print(f"Seed {seed} установлен")

SEED = 42
set_seed(SEED)

Seed 42 установлен


In [4]:
CONFIG = {
    "device": "cuda",
    "max_new_tokens": 256,
    "temperature": 0.5,
    "top_p": 0.5,
    "do_sample": True,
    "seed": SEED,
    "models": {
        "qwen": {
            "model_name": "Qwen/Qwen2.5-1.5B-Instruct",
        },
        "meno-tiny": {
            "model_name": "bond005/meno-tiny-0.1",
        }
    }
}


# Загрузка модели

In [5]:
BASE_PROMPT = """
Ты — эксперт по упрощению русских текстов.

Задача:
Адаптируй исходный текст на русском языке до целевого уровня сложности {target_level} по шкале CEFR.

ВАЖНЕЙШЕЕ ПРАВИЛО — НАРУШЕНИЕ НЕДОПУСТИМО:
Ты работаешь ТОЛЬКО с тем фрагментом текста, который дан.
ЗАПРЕЩЕНО добавлять любые действия, события, мысли, последствия или оценки, которых нет в исходном тексте.
Не развивай сюжет. Не завершай мысль автора. Не придумывай, что было "потом", "тогда", "после этого".
Если исходный текст обрывается — твой ответ тоже должен оборваться. Твоя задача — упростить язык, а не пересказать историю.

Требования:
- Итоговый текст должен быть на русском языке. Латиница возможна только в именах собственных.
- Текст должен быть максимально близким по содержанию к исходному. Не добавляй и не удаляй важную информацию.
- Соблюдай грамматические, орфографические и пунктуационные нормы русского языка.
- Выведи только итоговый адаптированный текст. Никаких вступлений, комментариев, пояснений или повторений исходного текста.
- Запрещены циклические повторы. Не повторяй одно и то же слово или фразу несколько раз подряд.
- Варьируй структуру предложений: чередуй простые и сложные предложения (в рамках допустимого для целевого уровня), меняй порядок слов для естественности звучания.
- Следи за разнообразием лексики: не используй одно и то же слово в соседних предложениях, если это не ключевой термин.

{additional_instructions}

Исходный текст:
{text}
"""

In [6]:
ADDITIONAL_INSTRUCTIONS = {
    "simplify_more": """
Упрости текст сильнее:
- Замени сложные слова на простые
- Разделяй длинные предложения
- Избегай причастных и деепричастных оборотов
- Используй простые грамматические конструкции
""",

    "simplify_less": """
Не делай текст слишком примитивным:
- Сохрани все важные детали
- Не сокращай предложения слишком сильно
- Не удаляй ключевые элементы содержания
"""
}

In [7]:
LOADED_MODELS = {}

In [8]:
def load_model(model_key):
    """
    Загружает модель один раз и сохраняет в глобальном словаре.
    При повторном вызове возвращает уже загруженную модель.
    """
    if model_key in LOADED_MODELS:
        print(f"Модель {model_key} уже загружена, используется кэш")
        return LOADED_MODELS[model_key]

    print(f"Загрузка модели {model_key}...")
    model_info = CONFIG["models"][model_key]
    model_name = model_info["model_name"]

    tokenizer = AutoTokenizer.from_pretrained(model_name)

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float16,
        device_map="auto"
    )

    model.eval()

    model_data = {
        'tokenizer': tokenizer,
        'model': model,
        'name': model_name
    }

    LOADED_MODELS[model_key] = model_data
    print(f"Модель {model_key} успешно загружена")

    return model_data

In [9]:
def unload_model(model_key):
    """Выгружает модель из памяти"""
    if model_key in LOADED_MODELS:
        if 'model' in LOADED_MODELS[model_key]:
            del LOADED_MODELS[model_key]['model']
        if 'tokenizer' in LOADED_MODELS[model_key]:
            del LOADED_MODELS[model_key]['tokenizer']
        del LOADED_MODELS[model_key]

        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        print(f"Модель {model_key} выгружена")

In [10]:
def format_prompt(text, target_level="A2", additional_instructions_key=None):
    """
    Формирует полный промпт для генерации текста.
    """
    cefr_levels = ['A1', 'A2', 'B1', 'B2', 'C1', 'C2']
    if target_level not in cefr_levels:
        raise ValueError("Переданный уровень не соответствует шкале CEFR")

    additional_instructions = ""
    if additional_instructions_key and additional_instructions_key in ADDITIONAL_INSTRUCTIONS:
        additional_instructions = ADDITIONAL_INSTRUCTIONS[additional_instructions_key]

    prompt = BASE_PROMPT.format(
        target_level=target_level,
        additional_instructions=additional_instructions,
        text=text
    )

    return prompt.strip()

In [11]:
def generate(
    model_key,
    prompt,
    custom_config=None,
    **gen_kwargs
):
    if model_key not in LOADED_MODELS:
        raise ValueError(f"Модель {model_key} не загружена.")

    model_data = LOADED_MODELS[model_key]
    tokenizer = model_data['tokenizer']
    model = model_data['model']

    config_to_use = custom_config if custom_config else CONFIG

    generation_params = {
        'max_new_tokens': config_to_use["max_new_tokens"],
        'temperature': config_to_use["temperature"],
        'top_p': config_to_use["top_p"],
        'do_sample': config_to_use["do_sample"],
    }
    generation_params.update(gen_kwargs)

    with torch.no_grad():
        messages = [{"role": "user", "content": prompt}]

        formatted_prompt = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

        inputs = tokenizer(
            formatted_prompt,
            return_tensors="pt"
        ).to(model.device)

        outputs = model.generate(
            **inputs,
            **generation_params,
            pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id
        )

        input_length = inputs.input_ids.shape[1]
        generated_tokens = outputs[0][input_length:]
        generated_text = tokenizer.decode(
            generated_tokens,
            skip_special_tokens=True
        )

    return generated_text.strip()

In [13]:
# Загрузка моделей
# Выбираем нужную в зависимости от того, с какой хотим работать
load_model('qwen')
# load_model('meno-tiny')

Загрузка модели qwen...


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Модель qwen успешно загружена


{'tokenizer': Qwen2Tokenizer(name_or_path='Qwen/Qwen2.5-1.5B-Instruct', vocab_size=151643, model_max_length=131072, padding_side='right', truncation_side='right', special_tokens={'eos_token': '<|im_end|>', 'pad_token': '<|endoftext|>'}, added_tokens_decoder={
 	151643: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
 	151644: AddedToken("<|im_start|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
 	151645: AddedToken("<|im_end|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
 	151646: AddedToken("<|object_ref_start|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
 	151647: AddedToken("<|object_ref_end|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
 	151648: AddedToken("<|box_start|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
 	151649: AddedToken("<|

# Reward функция

In [21]:
!unzip ru_bert_cefr.zip

Archive:  ru_bert_cefr.zip
   creating: new_ru_bert_cefr/checkpoint-6615/
  inflating: new_ru_bert_cefr/checkpoint-6615/special_tokens_map.json  
  inflating: new_ru_bert_cefr/checkpoint-6615/optimizer.pt  
  inflating: new_ru_bert_cefr/checkpoint-6615/tokenizer_config.json  
  inflating: new_ru_bert_cefr/checkpoint-6615/model.safetensors  
  inflating: new_ru_bert_cefr/checkpoint-6615/tokenizer.json  
  inflating: new_ru_bert_cefr/checkpoint-6615/vocab.txt  
  inflating: new_ru_bert_cefr/checkpoint-6615/trainer_state.json  
  inflating: new_ru_bert_cefr/checkpoint-6615/scheduler.pt  
  inflating: new_ru_bert_cefr/checkpoint-6615/config.json  
  inflating: new_ru_bert_cefr/checkpoint-6615/scaler.pt  
  inflating: new_ru_bert_cefr/checkpoint-6615/rng_state.pth  
  inflating: new_ru_bert_cefr/checkpoint-6615/training_args.bin  


In [22]:
classifier = AutoModelForSequenceClassification.from_pretrained('new_ru_bert_cefr/checkpoint-6615')
tokenizer_cls = AutoTokenizer.from_pretrained('new_ru_bert_cefr/checkpoint-6615')

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [23]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
classifier.to(device)
classifier.eval()

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(120138, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12

In [24]:
def level_diff(sum_text, goal_level):
  '''
  Принимает изначальный тект, текст после изменения, целевой уровень, возвращает разность уровней.
  '''
  #label2id
  class_to_numb = {'A1':0,
                   'A2':1,
                   'B1':2,
                   'B2':3,
                   'C1':4,
                   'C2':5}

  # предсказываем класс изменённого текста
  inputs = tokenizer_cls(sum_text, return_tensors='pt', truncation=True, padding=True)
  device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
  classifier.to(device)
  inputs = {k: v.to(device) for k, v in inputs.items()}
  outputs = classifier(**inputs)

  sum_level = torch.argmax(outputs.logits, dim=-1).item()
  base_level = class_to_numb[goal_level]

  # возвращаем разность классов, отрицательная, если класс суммаризации выше исходного
  return sum_level-base_level

In [25]:
cache_dir = "/content/bert_score_cache"
os.makedirs(cache_dir, exist_ok=True)
os.environ['HF_HOME'] = cache_dir
os.environ['TRANSFORMERS_CACHE'] = cache_dir
os.environ['HUGGINGFACE_HUB_CACHE'] = cache_dir
os.environ['TORCH_HOME'] = cache_dir

In [26]:
# для bertscore модель, чтобы один раз скачать
bert_scorer = BERTScorer(
    lang='ru',
    model_type='bert-base-multilingual-cased'
    )

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [27]:
def postpoc_rl(base_text:str, sum_text:str, goal_level:str, metric='BERTSCORE',
               level_weight=0.5):
  '''
  Принимает изначальный тект, текст после изменения, целевой уровень,
  метрику близости текстов (базово BERTSCORE),
  веса для вклада близости текстов и разности уровней.

  base_text:str -- изначальный текст
  sum_text:str -- текст после суммаризации/изменения уровня LLM
  goal_level:str -- желаемый уровень финального текста, от A1 до C2
  metric:str -- метрика близости текста, влияющая на reward: BERTSCORE, BLEU
  level_weight:float -- вес (не)совпадения желаемого и реального уровня
  текста LLM, от 0.0 до 1.0. Вес метрики вычисляется как 1.0 - level_weight

  Возвращает reward score [0, 1], где 1.0 -- лучший вариант, 0.0 -- худший
  '''

  if goal_level not in ['A1','A2', 'B1', 'B2', 'C1', 'C2']:
    raise ValueError(f'Неверно выбран целевой вариант сложности текста {goal_level}, он должен относиться к шкале CEFR')

  # сильно штрафуем за генерации длинных текстов
  if len(sum_text) > 1.5*(len(base_text)):
    return 0.0

  level_delta = abs(level_diff(sum_text, goal_level)) # разность между CEFR уровнями текстов [-5,5]
  max_level_diff = 5.0

  if level_delta < 0: # суммаризация сложнее нужного уровня
    level_normalized = level_delta/max_level_diff # [-1,0)
    level_weight*=1.5 # сильнее наказываем такие ошибки
  else:
    level_normalized = -level_delta/max_level_diff

  proximity_weight = 1.0 - level_weight
  prox_score = 0.0

  if metric == 'BERTSCORE':

    bert_precision, bert_recall, bert_f1 = bert_scorer.score([sum_text], [base_text])
    bert_norm = 1.0# уже отнормировано от 0 до 1
    prox_score = bert_f1.item()*bert_norm # базово берём за скор f1

  elif metric == 'BLEU':
    bleu_norm = 0.01
    bleu_score = sacrebleu.sentence_bleu(sum_text, [base_text]).score* bleu_norm
    prox_score = min(bleu_score, 1.00) # сглаживание
  else:
    raise ValueError(f'Неверно выбрана метрика {metric}, доступные варианты: BERTSCORE, BLEU')

  reward = level_weight * level_normalized + proximity_weight * prox_score

  return max(0.0, min(1.0, reward)) # от 0 до 1

# trl

In [ ]:
# Создаем датасет с сохранением информации о целевом уровне
train_dataset = pd.read_csv('train_dataset.csv')
raw_texts = list(train_dataset[train_dataset['cefr_level'] != 'A1']['text'])
target_levels_list = ['A1', 'A2', 'B1', 'B2', 'C1']

# Создаем список словарей: промпт, целевой_уровень, базовый_текст
prompts_with_targets = []
for target_level in target_levels_list:
    for text in raw_texts:
        prompt = format_prompt(text, target_level=target_level)
        prompts_with_targets.append({
            'prompt': prompt,
            'target_level': target_level,
            'base_text': text
        })

# Создаем датасет с дополнительными полями
dataset = Dataset.from_list(prompts_with_targets)

In [ ]:
def compute_reward_for_trainer(prompts, completions, base_text, target_level, **kwargs):
    '''
    Reward-функция в необходмом для трейнера формате.
    Возвращает значения нашей функции postpoc_rl, если в процессе выполнения не возникает ошибки.
    '''
    rewards = []
    for bt, tl, comp in zip(base_text, target_level, completions):
        try:
            rewards.append(
                postpoc_rl(
                    base_text=bt,
                    sum_text=comp,
                    goal_level=tl,
                    metric="BERTSCORE",
                    level_weight=0.35,
            	)
        	)
        except Exception as e:
            print(f"reward error: {e}")
            rewards.append(-1.0)
    return rewards

In [ ]:
# загрузка модели и токенизатора (выбираем квен или мено)
# model_data = load_model('meno-tiny')
model_data = load_model('qwen')
model_to_train = model_data['model']
tokenizer_to_train = model_data['tokenizer']

Модель qwen уже загружена, используется кэш


In [ ]:
# лора, чтобы памяти хватило
lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM"
)

In [ ]:
model_to_train = get_peft_model(model_to_train, lora_config)
model_to_train.print_trainable_parameters()
model_to_train.gradient_checkpointing_enable()

trainable params: 9,232,384 || all params: 1,552,946,688 || trainable%: 0.5945


In [ ]:
# аргументы обучения
training_args = GRPOConfig(
    output_dir="./qwen_simplifier_lora",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=5e-4,
    beta=0.2,
    max_completion_length=128,
    num_generations=4,
    logging_steps=50,
    save_steps=75,
    max_steps=150,
    report_to="none",
    temperature=0.3,
    save_total_limit=1,
    generation_kwargs={
        "repetition_penalty": 1.5,
        "no_repeat_ngram_size": 4,
        "max_new_tokens": 128,
        "top_p": 0.85,
    },
    fp16=True,
)

In [ ]:
# тренер
trainer = GRPOTrainer(
    model=model_to_train,
    reward_funcs=[compute_reward_for_trainer],
    args=training_args,
    train_dataset=dataset,
    processing_class=tokenizer_to_train,
)

In [ ]:
# обучение тренера
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.
Passing `generation_config` together with generation-related arguments=({'disable_compile'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


Step,Training Loss
50,10190.847500
100,75.400757
150,0.547121


TrainOutput(global_step=150, training_loss=3422.265126101176, metrics={'train_runtime': 3352.2591, 'train_samples_per_second': 0.358, 'train_steps_per_second': 0.045, 'total_flos': 0.0, 'train_loss': 3422.265126101176})

In [ ]:
# Создаем ZIP-архив папки
folder_to_zip = "./qwen_simplifier_lora"
zip_name = "qwen_simplifier_lora.zip"

with zipfile.ZipFile(zip_name, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for root, dirs, files in os.walk(folder_to_zip):
        for file in files:
            file_path = os.path.join(root, file)
            arcname = os.path.relpath(file_path, os.path.dirname(folder_to_zip))
            zipf.write(file_path, arcname)

print(f"Архив создан: {zip_name}")

# Скачиваем
colab_files.download(zip_name)

Архив создан: qwen_simplifier_lora.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Тестирование на всём датасете

In [16]:
# загружаем тестовый датасет, на котором будем измерять качество
test_dataset = pd.read_csv('test_dataset.csv')

In [14]:
def test_model(texts, target_levels_idx, model_key='qwen-finetuned'):
    '''
    Функция, которая для каждого текста в датасете генерирует упрощённый вариант.
    Возвращает датафрейм:
      исходный текст,
      упрощённый текст,
      целевой уровень,
      предсказанный бертом уровень.
    '''

    simplified_texts_df = pd.DataFrame(columns=['original_text',
                                                'simplified_text',
                                                'target_level',
                                                'predicted_level'])

    idx_to_class = {0:'A1', 1:'A2', 2:'B1', 3:'B2', 4:'C1', 5:'C2'}

    for text, target_level_idx in tqdm(zip(texts, target_levels_idx),
                                       total=len(texts),
                                       position=0,
                                       leave=True):
        # Маппинг индексов в уровни CEFR
        target_level = idx_to_class[target_level_idx]

        prompt = format_prompt(text, target_level=target_level)
        # получаем упрощённый текст
        result = generate(
            model_key=model_key,
            prompt=prompt,
            temperature=0.3,
            max_new_tokens=256,
            do_sample=True,
            top_p=0.9,
            repetition_penalty=1.2
        )

        # получаем уровень от берта (для упрощённого текста)
        inputs_cls = tokenizer_cls(result, return_tensors='pt', truncation=True, padding=True, max_length=512)
        inputs_cls = {k: v.to(device) for k, v in inputs_cls.items()}

        with torch.no_grad():
            outputs_cls = classifier(**inputs_cls)
            pred_level_idx = torch.argmax(outputs_cls.logits, dim=-1).item()
        # Маппинг индексов в уровни CEFR
        predicted_level = idx_to_class[pred_level_idx]

        new_row = pd.DataFrame({
            'original_text': [text],
            'simplified_text': [result],
            'target_level': [target_level],
            'predicted_level': [predicted_level]
        })

        simplified_texts_df = pd.concat([simplified_texts_df, new_row], ignore_index=True)

    return simplified_texts_df

In [17]:
# для ускорения каждому тексту присвоим случайное значение уровня для упрощения
n_total = test_dataset.shape[0]
n_unique = 6
base_count = n_total // n_unique
remainder = n_total % n_unique

target_levels_idx = list(range(6)) * base_count

target_levels_idx.extend(range(remainder))

np.random.shuffle(target_levels_idx)

## QWEN

In [ ]:
test_texts = list(test_dataset['text'])

simplified_texts_qwen = test_model(test_texts,
                                   target_levels_idx,
                                   model_key='qwen-finetuned')

simplified_texts_qwen.to_csv('simplified_texts_qwen.csv', index=False, encoding='utf-8')

colab_files.download('simplified_texts_qwen.csv')

100%|██████████| 2645/2645 [2:53:45<00:00,  3.94s/it]


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
for index, row in simplified_texts_qwen.sample(5).iterrows():
    print('ОРИГИНАЛ:')
    print(row['original_text'])
    print(f'УПРОЩЁННЫЙ ТЕКСТ (УРОВЕНЬ: {row["predicted_level"]}, ЦЕЛЬ: {row["target_level"]}):')
    print(row['simplified_text'])
    print('='*50)

ОРИГИНАЛ:
Они хотели, чтобы всё оставалось чётким и холодным, без чувств и переживаний.
УПРОЩЁННЫЙ ТЕКСТ (УРОВЕНЬ: A1, ЦЕЛЬ: A1):
Они хотели, чтобы мир был спокоен и неподвижен, как камень.
ОРИГИНАЛ:
В городе проживало много богатых людей, молодых офицеров и знатных дам, которые наслаждались отдыхом, гуляли по благоустроенным бульварам, пили минеральную воду и обсуждали моду, любовь и последние новости.
УПРОЩЁННЫЙ ТЕКСТ (УРОВЕНЬ: A2, ЦЕЛЬ: A2):
Многие жители города были состоятельными людьми, среди них были юные военные и высокопоставленные особы, которым доставались приятные отпуска. Они могли прогуливаться по чистым набережным, выпивать минеральное молоко и беседовать о моде, amour et les dernières nouvelles du monde.
ОРИГИНАЛ:
Не забудьте надеть фрак.
УПРОЩЁННЫЙ ТЕКСТ (УРОВЕНЬ: B1, ЦЕЛЬ: B2):
Наденьте фрак!
ОРИГИНАЛ:
я уж и лег сегодня, с десяти часов, чтоб уж совсем не вставать до самого того времени, да вот раздумал и встал еще раз, чтобы к вам идти…
УПРОЩЁННЫЙ ТЕКСТ (УРОВЕНЬ: B2,

In [ ]:
matches = (simplified_texts_qwen['predicted_level'] == simplified_texts_qwen['target_level']).sum()
print(f"Accuracy: {matches / simplified_texts_qwen.shape[0]}")

Accuracy: 0.17996219281663517


In [ ]:
not_matched = simplified_texts_qwen[simplified_texts_qwen['predicted_level'] != simplified_texts_qwen['target_level']]

In [ ]:
not_matched = not_matched.copy()

class_to_idx = {'A1':0, 'A2':1, 'B1':2, 'B2':3, 'C1':4, 'C2':5}

not_matched['target_level_idx'] = not_matched['target_level'].map(class_to_idx)
not_matched['predicted_level_idx'] = not_matched['predicted_level'].map(class_to_idx)
not_matched['level_diff'] = not_matched['predicted_level_idx'] - not_matched['target_level_idx']

not_matched['level_diff'].value_counts()

,count
level_diff,
1,355
-1,350
-2,298
-3,275
2,237
-4,202
3,156
4,143
-5,80


In [ ]:
not_matched[not_matched['level_diff'] == 5].loc[248]

,248
original_text,Но все было бесплодно.
simplified_text,Ничего не получилось.
target_level,A1
predicted_level,C2
target_level_idx,0
predicted_level_idx,5
level_diff,5


## MENO-TINY

In [18]:
!unzip meno_simplifier_lora.zip

Archive:  meno_simplifier_lora.zip
  inflating: meno_simplifier_lora/README.md  
  inflating: meno_simplifier_lora/checkpoint-150/optimizer.pt  
  inflating: meno_simplifier_lora/checkpoint-150/chat_template.jinja  
  inflating: meno_simplifier_lora/checkpoint-150/scaler.pt  
  inflating: meno_simplifier_lora/checkpoint-150/tokenizer.json  
  inflating: meno_simplifier_lora/checkpoint-150/adapter_model.safetensors  
  inflating: meno_simplifier_lora/checkpoint-150/tokenizer_config.json  
  inflating: meno_simplifier_lora/checkpoint-150/scheduler.pt  
  inflating: meno_simplifier_lora/checkpoint-150/adapter_config.json  
  inflating: meno_simplifier_lora/checkpoint-150/training_args.bin  
  inflating: meno_simplifier_lora/checkpoint-150/README.md  
  inflating: meno_simplifier_lora/checkpoint-150/rng_state.pth  
  inflating: meno_simplifier_lora/checkpoint-150/trainer_state.json  
  inflating: meno_simplifier_lora/checkpoint-150/ref/adapter_model.safetensors  
  inflating: meno_simplifi

In [19]:
# загрузка базовой модели (квен)
base_model_name = CONFIG['models']['meno-tiny']['model_name']
tokenizer = AutoTokenizer.from_pretrained(base_model_name)
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

config.json:   0%|          | 0.00/717 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/499 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/286 [00:00<?, ?B/s]

In [20]:
model_path = "./meno_simplifier_lora/checkpoint-150"
model = PeftModel.from_pretrained(base_model, model_path)
model.eval()

# if tokenizer.pad_token is None:
#     tokenizer.pad_token = tokenizer.eos_token

# Добавляем модель в LOADED_MODELS для использования с generate()
LOADED_MODELS['meno-tiny-finetuned'] = {
    'tokenizer': tokenizer,
    'model': model,
    'name': 'meno-tiny-finetuned'
}

In [ ]:
test_texts = list(test_dataset['text'])

simplified_texts_meno = test_model(test_texts,
                                   target_levels_idx,
                                   model_key='meno-tiny-finetuned')

simplified_texts_meno.to_csv('simplified_texts_meno.csv', index=False, encoding='utf-8')

In [30]:
colab_files.download('simplified_texts_meno.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [34]:
for index, row in simplified_texts_meno.sample(5).iterrows():
    print('ОРИГИНАЛ:')
    print(row['original_text'])
    print(f'УПРОЩЁННЫЙ ТЕКСТ (УРОВЕНЬ: {row["predicted_level"]}, ЦЕЛЬ: {row["target_level"]}):')
    print(row['simplified_text'])
    print('='*50)

ОРИГИНАЛ:
Князь немедленно хотел поворотить назад к себе, в гостиницу; даже повернулся и потел; но чрез минуту остановился, обдумал и воротился опять по прежней дороге.
УПРОЩЁННЫЙ ТЕКСТ (УРОВЕНЬ: B2, ЦЕЛЬ: B2):
Князь хотел повернуть назад к себе, в гостиницу, но через минуту остановился, обдумал и воротился опять по прежней дороге.
ОРИГИНАЛ:
Люди проходили через узкие переулки, переговаривались шёпотом и вдруг начинали петь старые песни, которые не всегда можно было разобрать.
УПРОЩЁННЫЙ ТЕКСТ (УРОВЕНЬ: A1, ЦЕЛЬ: A2):
Люди проходили через узкие переулки, переговаривались шепотом и вдруг начали петь старые песни, которые не всегда могли разобрать.
ОРИГИНАЛ:
Однажды Грей услышал о маленькой деревне у моря, где живёт девушка по имени Ассоль, которая верит в чудо и ждёт корабль с алыми парусами.
УПРОЩЁННЫЙ ТЕКСТ (УРОВЕНЬ: A1, ЦЕЛЬ: A2):
Однажды Грей услышал о маленькой деревне у моря, где живёт девушка по имени Ассоль, которая верит в чудо и ждёт корабль с алыми парусами.
ОРИГИНАЛ:
Зазелен

In [35]:
matches = (simplified_texts_meno['predicted_level'] == simplified_texts_meno['target_level']).sum()
print(f"Accuracy: {matches / simplified_texts_meno.shape[0]}")

Accuracy: 0.1659735349716446


In [36]:
not_matched = simplified_texts_meno[simplified_texts_meno['predicted_level'] != simplified_texts_meno['target_level']]

In [37]:
not_matched = not_matched.copy()

class_to_idx = {'A1':0, 'A2':1, 'B1':2, 'B2':3, 'C1':4, 'C2':5}

not_matched['target_level_idx'] = not_matched['target_level'].map(class_to_idx)
not_matched['predicted_level_idx'] = not_matched['predicted_level'].map(class_to_idx)
not_matched['level_diff'] = not_matched['predicted_level_idx'] - not_matched['target_level_idx']

not_matched['level_diff'].value_counts()

,count
level_diff,
1,356
-1,354
-2,315
-3,280
2,248
3,192
-4,179
4,115
-5,91
